In [ ]:
from matplotlib import pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from adjustText import adjust_text
import seaborn as sns

from types import NoneType
import os
from os import PathLike
from pathlib import Path

import pandas as pd
import numpy as np

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

from utils.color_palettes import black_red_pastel_palette
from utils import GENES_OF_INTEREST, SCREEN_GENES, FIBROSIS_GENES
from RNA_Seq import RNASeq_Data

# from scipy.stats import gaussian_kde


In [ ]:
SAMPLE_FILES = {
    "LX2": Path("/data_gilbert3/asathler/datasets/0226_DC_RNA_seq/2026.02.23_DC_LX-2_TGFB/HM8D5T-expression-matrix.tsv"),
    "pHSC": Path("/data_gilbert3/asathler/datasets/0226_DC_RNA_seq/2026.05.23_DC_phHSC_TGFB/2HCQCH-expression-matrix.tsv"),
}
SAMPLE_INDEXES = {
    "Ctrl": [0, 1],
    "TGF\u03b2": [2, 3]
}
OUTPUT_DIR = Path("/data_gilbert3/asathler/datasets/0226_DC_RNA_seq/")

In [ ]:
TEST_GENE = GENES_OF_INTEREST[0]

In [ ]:
data = RNASeq_Data(
    filepath=SAMPLE_FILES["LX2"],
    sep="\t",
    index_col_name="gene_name",
    sample_number_cat_name="treatment",
    sample_number_cats={
        "Ctrl": [1, 2],
        "TGFb": [3, 4]
    },
    cache_dir=OUTPUT_DIR / ".deseq2_cache"
)

data.add_csv(
    filepath=SAMPLE_FILES["pHSC"], sep="\t",
    sample_number_cat_name="treatment",
    sample_number_cats={
        "Ctrl": [1, 2],
        "TGFb": [3, 4]
    },
    category_name="sample",
    old_data_category="LX2",
    new_data_category="pHSC"
)

### Visualize Key Genes (bar graph)
*_TODO_*: Fix bar graph

In [ ]:
mpl.style.use("ggplot")
NROWS, NCOLS = 5, 4
fig, axes = plt.subplots(nrows = NROWS, ncols = NCOLS, figsize=(8.5,11), sharex=False, sharey=False)
# START, END = None, NROWS*NCOLS # to define how many genes are being graphed
START, END = NROWS * NCOLS, None
axes = axes.reshape((axes.shape[0] * axes.shape[1]))
for adx, gene in enumerate(GENES_OF_INTEREST[START:END]):
    these_data = total_merged.loc[total_merged['gene_name'] == gene]
    this_axis = axes[adx]
    sns.barplot(
        these_data, x="cell_type", y="value", 
        hue="sample", palette=black_red_pastel_palette, 
        edgecolor="black", linewidth=0.5,
        ax=this_axis, legend=True if adx==0 else False,
        errorbar="sd", capsize=0, err_kws={"linewidth": 1},
    )
    this_axis.set_title(gene)
    this_axis.set_xlabel(None)
    if np.all(these_data["mean"]) == 0: this_axis.set_ylim(0)
    if adx % NCOLS == 0: this_axis.set_ylabel("CPM")
    else: this_axis.set_ylabel(None)
    if adx == 0: this_axis.legend(loc=(0, 1.2))

this_legend = "RNA-seq counts per million (CPM) for selected genes in LX2 and primay hepatic\nstellate cells (pHSCs) with and without TGF\u03b2 treatment."
fig.text(x=0.2, y=0.955, s=this_legend, fontdict={"size": 12, "style": "italic"})
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(OUTPUT_DIR, f"RNASeq_Gene_Exp_Analysis_{START}-{END}.pdf"))

### Scatterplot All Genes

#### Usage

##### Single panel:
```
data._setup_comparison_table("treatment", filters={2: "pHSC"})
data._calculate_comparison_p_values()
ax, hits = data.plot_comparison_density(top_n=20)
plt.show()
```

##### Multi-panel with a shared color scale:
```
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), layout="constrained")
shared_norm = mcolors.LogNorm(vmin=1, vmax=150)
for ax, sample in zip(axes, ["LX2", "pHSC"]):
    data._setup_comparison_table("treatment", filters={2: sample})
    data._calculate_comparison_p_values()
    data.plot_comparison_density(ax=ax, top_n=20, norm=shared_norm,
                                 colorbar=(sample == "pHSC"))
    data.close_comparison()
plt.show()
```

In [ ]:
data.close_comparison()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12), layout="constrained")
shared_norm = mcolors.LogNorm(vmin=1, vmax=150)
for rdx, sample in zip(np.arange(len(axes)), ["LX2", "pHSC"]):
    data._setup_comparison_table("treatment", filters={2: sample})
    data._calculate_comparison_p_values()
    for cdx in np.arange(len(axes[rdx])):
        if cdx == 0:
            data.plot_comparison_density(
                ax=axes[rdx][cdx], top_n=20, highlight_color="b", 
                norm=shared_norm, colorbar=False, log_ticks=True, highlight_pointsize=20,
                label_fontsize=10
            )
        elif cdx == 1:
            data.plot_comparison_density(
                ax=axes[rdx][cdx], genes=SCREEN_GENES, highlight_color=["aqua", "violet", "green"],
                norm=shared_norm, colorbar=False, log_ticks=True, highlight_pointsize=20,
                label_fontsize=10
            )
        elif cdx == 2:
            data.plot_comparison_density(
                ax=axes[rdx][cdx], genes=FIBROSIS_GENES, highlight_color="k",
                norm=shared_norm, colorbar=False, log_ticks=True, highlight_pointsize=20,
                label_fontsize=10
            )
    data.close_comparison()
plt.savefig(OUTPUT_DIR / "cell-by-treatment.png")
plt.savefig(OUTPUT_DIR / "cell-by-treatment.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12), layout="constrained")
shared_norm = mcolors.LogNorm(vmin=1, vmax=150)
for rdx, sample in zip(np.arange(len(axes)), ["Ctrl", "TGFb"]):
    data._setup_comparison_table("sample", filters={1: sample})
    data._calculate_comparison_p_values()
    for cdx in np.arange(len(axes[rdx])):
        if cdx == 0:
            data.plot_comparison_density(
                ax=axes[rdx][cdx], top_n=20, highlight_color="b", 
                norm=shared_norm, colorbar=False, log_ticks=True, highlight_pointsize=20,
                label_fontsize=10
            )
        elif cdx == 1:
            data.plot_comparison_density(
                ax=axes[rdx][cdx], genes=SCREEN_GENES, highlight_color=["aqua", "violet", "green"],
                norm=shared_norm, colorbar=False, log_ticks=True, highlight_pointsize=20,
                label_fontsize=10
            )
        elif cdx == 2:
            data.plot_comparison_density(
                ax=axes[rdx][cdx], genes=FIBROSIS_GENES, highlight_color="k",
                norm=shared_norm, colorbar=False, log_ticks=True, highlight_pointsize=20,
                label_fontsize=10
            )
    data.close_comparison()
plt.savefig(OUTPUT_DIR / "treatment-by-cell.png")
plt.savefig(OUTPUT_DIR / "treatment-by-cell.svg")
plt.show()

In [ ]:
import json

In [ ]:
js = {}
df = {}
for j in sorted(data.cache_dir.glob("*.gz")):
    df[j.name.split(".")[0]] = pd.read_csv(j, compression="gzip")
for j in sorted(data.cache_dir.glob("*.json")):
    js[j.name.split(".")[0]] = json.loads(j.read_text())

In [ ]:
comp_df = data.list_cached_comparisons()

In [ ]:
comp_df.head()

In [ ]:
comp_df = comp_df[comp_df["filter_1"].isna()][["key", "filter_2"]]

In [ ]:
from matplotlib import colors as mcolors

In [ ]:
%matplotlib inline

In [ ]:
comp_df.to_csv(data.cache_dir / "t2fc_hash.csv")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
ax0 = axs[0]
ax = axs[1]
ax2 = axs[2]

X=df[comp_df.key[0]].dropna()
Y=df[comp_df.key[1]].dropna()
X_index = X.index.intersection(Y.index)
X=X.loc[X_index,:]
Y=Y.loc[X_index,:]
X = X.sort_values("gene_name")
Y = Y.sort_values("gene_name")

X["delta"] = X["log2FoldChange"] - Y["log2FoldChange"]
X["delta"] = X["delta"].abs()
delta = X.sort_values("delta", ascending=False)[:7]["gene_name"]

X[f"x_greater"] = X[~X["gene_name"].isin(delta)]["log2FoldChange"].abs()
x_greater = X.sort_values("x_greater", ascending=False)[:7]["gene_name"]

X[f"y_greater"] = Y[~Y["gene_name"].isin(np.concatenate([delta, x_greater]))]["log2FoldChange"]
X["y_greater"] = X["y_greater"].abs()
y_greater = X.sort_values("y_greater", ascending=False)[:7]["gene_name"]

"""
Plotting 2d histograms
"""
cmap = mpl.colormaps["rocket_r"].copy()  # copy so the global registry isn't modified
cmap.set_bad(alpha=0)
_, _, _, im0 = ax0.hist2d(
    x=X["log2FoldChange"], y=Y["log2FoldChange"],
    bins=(100,100), cmin=1, cmap=cmap, norm=mcolors.LogNorm(1, 1000)
)

_, _, _, im = ax.hist2d(
    x=X["log2FoldChange"], y=Y["log2FoldChange"],
    bins=(100,100), cmin=1, cmap=cmap, norm=mcolors.LogNorm(1, 1000)
)

_, _, _, im2 = ax2.hist2d(
    x=X["log2FoldChange"], y=Y["log2FoldChange"],
    bins=(100,100), cmin=1, cmap=cmap, norm=mcolors.LogNorm(1, 1000)
)

"""
Plotting axlines
"""
ax0.axline((0, 0), slope=1, color="0.5", lw=1, ls="--")
ax.axline((0, 0), slope=1, color="0.5", lw=1, ls="--")
ax2.axline((0, 0), slope=1, color="0.5", lw=1, ls="--")

"""
PLOTTING SCATTER
"""
# ax0

HIGHLIGHT_GENES = {
    "Highest Delta": delta,
    "Highest LX2": x_greater,
    "Highest phHSC": y_greater,
}

for color, (key, genes) in zip(["aqua", "violet", "green"], HIGHLIGHT_GENES.items()):
    x_scatter = X.loc[X["gene_name"].isin(HIGHLIGHT_GENES[key]), :]
    y_scatter = Y.loc[Y["gene_name"].isin(HIGHLIGHT_GENES[key]), :]
    ax0.scatter(
        x=x_scatter["log2FoldChange"],
        y=y_scatter["log2FoldChange"],
        c=color, edgecolors="w", label=key
    )

# ax
for color, (key, genes) in zip(["aqua", "violet", "green"], SCREEN_GENES.items()):
    x_scatter = X.loc[X["gene_name"].isin(SCREEN_GENES[key]), :]
    y_scatter = Y.loc[Y["gene_name"].isin(SCREEN_GENES[key]), :]
    ax.scatter(
        x=x_scatter["log2FoldChange"],
        y=y_scatter["log2FoldChange"],
        c=color, edgecolors="w", label=key
    )

# ax2
x_scatter = X.loc[X["gene_name"].isin(FIBROSIS_GENES), :]
y_scatter = Y.loc[Y["gene_name"].isin(FIBROSIS_GENES), :]
ax2.scatter(
    x=x_scatter["log2FoldChange"],
    y=y_scatter["log2FoldChange"],
    c="k", edgecolors="w"
)

"""
Getting texts
"""
texts0 = []
genes0 = []
for screen_key, these_genes in HIGHLIGHT_GENES.items():
    for gene in these_genes:
        if gene not in X["gene_name"].to_list(): continue
        str_x = X.loc[X["gene_name"] == gene, "log2FoldChange"].item()
        str_y = Y.loc[Y["gene_name"] == gene, "log2FoldChange"].item()
        # p_str = "NA" if pd.isna(p) else f"{p:.0e}"
        texts0.append(ax0.text(
            str_x, str_y, gene, fontsize=10,
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
        ))

texts = []
genes = []
for screen_key, these_genes in SCREEN_GENES.items():
    for gene in these_genes:
        if gene not in X["gene_name"].to_list(): continue
        str_x = X.loc[X["gene_name"] == gene, "log2FoldChange"].item()
        str_y = Y.loc[Y["gene_name"] == gene, "log2FoldChange"].item()
        # p_str = "NA" if pd.isna(p) else f"{p:.0e}"
        texts.append(ax.text(
            str_x, str_y, gene, fontsize=10,
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
        ))

texts2 = []
for gene in FIBROSIS_GENES:
    if gene not in X["gene_name"].to_list(): continue
    str_x = X.loc[X["gene_name"] == gene, "log2FoldChange"].item()
    str_y = Y.loc[Y["gene_name"] == gene, "log2FoldChange"].item()
    # p_str = "NA" if pd.isna(p) else f"{p:.0e}"
    texts2.append(ax2.text(
        str_x, str_y, gene, fontsize=10,
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
    ))

"""
Final prep
"""
ax0.legend()
ax.legend()

ax0.set_title("20 largest differential fold change genes")
ax.set_title("Highlighted genes from DC screen")
ax2.set_title("Fibrosis genes")

ax0.set_xlabel(f"{comp_df.filter_2[0]} TGFb/Ctrl L2FC")
ax0.set_ylabel(f"{comp_df.filter_2[1]} TGFb/Ctrl L2FC")
ax.set_xlabel(f"{comp_df.filter_2[0]} TGFb/Ctrl L2FC")
ax.set_ylabel(f"{comp_df.filter_2[1]} TGFb/Ctrl L2FC")
ax2.set_xlabel(f"{comp_df.filter_2[0]} TGFb/Ctrl L2FC")
ax2.set_ylabel(f"{comp_df.filter_2[1]} TGFb/Ctrl L2FC")

# fig.canvas.draw()  # finalize layout (aspect, colorbar, ticks) before adjusting


"""
Adjusting text
"""
adjust_text(
    texts0,
    avoid_self=True,
    x=X.loc[X["gene_name"].isin(genes0), "log2FoldChange"].to_list(), 
    y=Y.loc[Y["gene_name"].isin(genes0), "log2FoldChange"].to_list(),  # points to avoid
    ax=ax0, expand=(2, 2), force_text=(0.5, 0.5), force_static=(0.5,0.5),
    arrowprops=dict(arrowstyle="-", color="0.3", lw=1),
)

adjust_text(
    texts,
    avoid_self=True,
    x=X.loc[X["gene_name"].isin(genes), "log2FoldChange"].to_list(), 
    y=Y.loc[Y["gene_name"].isin(genes), "log2FoldChange"].to_list(),  # points to avoid
    ax=ax, expand=(2, 2), force_text=(0.5, 0.5), force_static=(0.5,0.5),
    arrowprops=dict(arrowstyle="-", color="0.3", lw=1),
)

adjust_text(
    texts2,
    avoid_self=True,
    x=X.loc[X["gene_name"].isin(FIBROSIS_GENES), "log2FoldChange"].to_list(), 
    y=Y.loc[Y["gene_name"].isin(FIBROSIS_GENES), "log2FoldChange"].to_list(),  # points to avoid
    ax=ax2, expand=(2, 2), force_text=(0.5, 0.5), force_static=(0.5,0.5),
    arrowprops=dict(arrowstyle="-", color="0.3", lw=1),
)
"""
Showing plot, saving
"""
plt.savefig(OUTPUT_DIR / "log2fc_highlights.png")
plt.savefig(OUTPUT_DIR / "log2fc_highlights.svg")
plt.show()